# Semantic Network with Inheritance

### Experiment 1: Knowledge Representation - Semantic Network with Inheritance

This notebook implements a semantic network of triples and answers queries using property inheritance along `is-a` links, including exception handling.

In [9]:
from collections import defaultdict
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

kb_triples = [
    ("shark", "is-a", "fish"),
    ("fish", "is-a", "animal"),
    ("bird", "is-a", "animal"),
    ("penguin", "is-a", "bird"),
    ("animal", "has", "skin"),
    ("bird", "can", "fly"),
    ("penguin", "cannot", "fly"),
    ("shark", "has", "gills"),
    ("fish", "has", "scales"),
    ("salmon", "is-a", "fish"),
    ("bat", "is-a", "mammal"),
    ("mammal", "is-a", "animal"),
    ("bat", "can", "fly"),
]

by_subject = defaultdict(list)
for s, r, o in kb_triples:
    by_subject[s].append((r, o))


In [10]:
def inheritance_chain(node):
    chain = [node]
    visited = {node}
    current = node
    while True:
        parents = [o for (r, o) in by_subject.get(current, []) if r == "is-a"]
        if not parents:
            break
        parent = parents[0]
        if parent in visited:
            break
        chain.append(parent)
        visited.add(parent)
        current = parent
    return chain

def query(node, relation):
    chain = inheritance_chain(node)
    for level in chain:
        facts = by_subject.get(level, [])
        for r, o in facts:
            if r == "cannot" and relation == "can" and o:
                return f"cannot {o} (from {level})"
        for r, o in facts:
            if r == relation:
                return f"{o} (from {level})"
    return "unknown"

print("Inheritance chain for penguin:", inheritance_chain("penguin"))
print("Query penguin can ->", query("penguin", "can"))
print("Query shark has ->", query("shark", "has"))

Inheritance chain for penguin: ['penguin', 'bird', 'animal']
Query penguin can -> cannot fly (from penguin)
Query shark has -> gills (from shark)


In [11]:
queries = [
    ("penguin", "can"),
    ("bird", "can"),
    ("shark", "has"),
    ("salmon", "has"),
    ("bat", "can"),
    ("penguin", "has"),
]
for node, rel in queries:
    print(f"query({node}, {rel}) -> {query(node, rel)}")


query(penguin, can) -> cannot fly (from penguin)
query(bird, can) -> fly (from bird)
query(shark, has) -> gills (from shark)
query(salmon, has) -> scales (from fish)
query(bat, can) -> fly (from bat)
query(penguin, has) -> skin (from animal)


In [16]:
kb_triples.extend([
    ("ostrich", "is-a", "bird"),
    ("ostrich", "can", "run_fast"),
])
by_subject.clear()
for s, r, o in kb_triples:
    by_subject[s].append((r, o))

print("Inheritance chain for ostrich:", inheritance_chain("ostrich"))
print("Query ostrich can ->", query("ostrich", "can"))
print("Query ostrich has ->", query("ostrich", "has"))


Inheritance chain for ostrich: ['ostrich', 'bird', 'animal']
Query ostrich can -> cannot fly (from ostrich)
Query ostrich has -> skin (from animal)


In [13]:
def plot_network(filename):
    G = nx.DiGraph()
    for s, r, o in kb_triples:
        if r == "is-a":
            G.add_edge(s, o, label="is-a")
        else:
            fact_node = f"{s}.{r}.{o}"
            G.add_node(fact_node, fact=True)
            G.add_edge(s, fact_node, label=r)
    pos = nx.spring_layout(G, seed=1)
    plt.figure(figsize=(10, 8))
    normal_nodes = [n for n, d in G.nodes(data=True) if not d.get('fact')]
    fact_nodes = [n for n, d in G.nodes(data=True) if d.get('fact')]
    nx.draw_networkx_nodes(G, pos, nodelist=normal_nodes, node_color='lightblue', node_size=1400)
    nx.draw_networkx_nodes(G, pos, nodelist=fact_nodes, node_color='lightgreen', node_size=800)
    nx.draw_networkx_labels(G, pos, font_size=9)
    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle='->', arrowsize=12)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=8)
    plt.title("Semantic Network Graph (triples)")
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    print(f"Saved plot: {filename}")

def plot_chain(node, filename):
    chain = inheritance_chain(node)
    plt.figure(figsize=(6, 2 + 0.8 * len(chain)))
    G = nx.DiGraph()
    for i in range(len(chain) - 1):
        G.add_edge(chain[i], chain[i + 1])
    pos = {n: (i, 0) for i, n in enumerate(chain)}
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=2000)
    nx.draw_networkx_labels(G, pos)
    nx.draw_networkx_edges(G, pos, arrows=True)
    plt.title(f"Inheritance chain for '{node}': " + " -> ".join(chain))
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    print(f"Saved plot: {filename}")

plot_network("network.png")
plot_chain("penguin", "chain_penguin.png")
plot_network("network_after_ostrich.png")
plot_chain("ostrich", "chain_ostrich.png")


Saved plot: network.png
Saved plot: chain_penguin.png
Saved plot: network_after_ostrich.png
Saved plot: chain_ostrich.png


In [14]:
frames = {
    "animal": {"has": "skin", "ako": None},
    "bird": {"ako": "animal", "can": "fly"},
    "penguin": {"ako": "bird", "cannot": "fly"},
    "shark": {"ako": "fish", "has": "gills"},
    "fish": {"ako": "animal", "has": "scales"},
}

def frame_chain(node):
    chain = [node]
    visited = {node}
    cur = node
    while True:
        parent = frames.get(cur, {}).get("ako")
        if not parent or parent in visited:
            break
        chain.append(parent)
        visited.add(parent)
        cur = parent
    return chain

def frame_query(node, relation):
    chain = frame_chain(node)
    for level in chain:
        frame = frames.get(level, {})
        if relation == "can" and "cannot" in frame:
            return f"cannot {frame['cannot']} (from {level})"
        if relation in frame:
            return f"{frame[relation]} (from {level})"
    return "unknown"

print("Frames-based KB demonstration:")
for q in [("penguin", "can"), ("bird", "can"), ("shark", "has"), ("fish", "has")]:
    print(f"frame_query({q[0]}, {q[1]}) -> {frame_query(q[0], q[1])}")


Frames-based KB demonstration:
frame_query(penguin, can) -> cannot fly (from penguin)
frame_query(bird, can) -> fly (from bird)
frame_query(shark, has) -> gills (from shark)
frame_query(fish, has) -> scales (from fish)


In [15]:
mp_kb = [
    ("flyingfish", "is-a", "fish"),
    ("flyingfish", "is-a", "flying_creature"),
    ("fish", "is-a", "animal"),
    ("flying_creature", "is-a", "animal"),
    ("animal", "has", "skin"),
    ("flying_creature", "can", "fly"),
    ("fish", "has", "scales"),
    ("flyingfish", "cannot", "fly"),
]
mp_by_subject = defaultdict(list)
for s, r, o in mp_kb:
    mp_by_subject[s].append((r, o))

def mp_chain(node):
    order = []
    visited = set()
    def dfs(n):
        if n in visited:
            return
        visited.add(n)
        order.append(n)
        parents = [o for (r, o) in mp_by_subject.get(n, []) if r == "is-a"]
        for p in parents:
            dfs(p)
    dfs(node)
    return order

def mp_query(node, relation):
    chain = mp_chain(node)
    for level in chain:
        facts = mp_by_subject.get(level, [])
        for r, o in facts:
            if r == "cannot" and relation == "can" and o:
                return f"cannot {o} (from {level})"
        for r, o in facts:
            if r == relation:
                return f"{o} (from {level})"
    return "unknown"

print("Multiple parents demo:")
print("mp_chain(flyingfish) ->", " -> ".join(mp_chain("flyingfish")))
print("mp_query(flyingfish, can) ->", mp_query("flyingfish", "can"))
print("mp_query(flyingfish, has) ->", mp_query("flyingfish", "has"))


Multiple parents demo:
mp_chain(flyingfish) -> flyingfish -> fish -> animal -> flying_creature
mp_query(flyingfish, can) -> cannot fly (from flyingfish)
mp_query(flyingfish, has) -> scales (from fish)


## Observations

- `query(penguin, can)` returns `cannot fly` even though `bird` can fly. This happens because the algorithm checks the most specific node first and gives precedence to a local exception before the inherited default.
- Exceptions must be checked before inherited values at every level of the chain, otherwise a more general rule could incorrectly override a more specific fact.
- This illustrates non-monotonic default reasoning: adding a more specific fact can invalidate a previously inherited conclusion.
- The frame-based version applies the same principle using an `ako` link and a slot-value structure.
- Multiple inheritance needs a consistent search order, such as depth-first left-to-right, to resolve conflicting parent paths predictably.